# RQFormer-2 Kaggle Full Workflow

Notebook này chạy **toàn bộ workflow trên Kaggle** cho RQFormer-2:

1. Copy source từ `/kaggle/input` sang `/kaggle/working`.
2. Link dataset đúng cấu trúc config gốc.
3. Link checkpoint gốc nếu có.
4. Setup môi trường OpenMMLab/MMRotate.
5. Train hoặc resume.
6. Test và lưu `predictions.pkl`.
7. Sinh chart/confusion nếu có đủ log/prediction.
8. Sinh 10 bbox visualization và 10 heatmap attention.
9. Ghi `summary_results.md`.
10. Zip output để tải về hoặc dùng resume.

Notebook không đổi kiến trúc source. Workflow chính gọi lại `tools/run_all_rqformer_visualize.sh` của repo, nên logic vẫn bám sát mã nguồn hiện tại.


## 0. Đánh giá fine-tuning trước khi chạy

Fine-tuning là phù hợp với source hiện tại vì pipeline train/test vẫn là pipeline gốc (`tools/train.py`, `tools/test.py`). Các module cải tiến được gắn vào decoder/RRoI attention, có tham số học được và đi qua optimizer như các layer bình thường.

Lưu ý quan trọng trên Kaggle:

- Nếu dùng checkpoint gốc, các layer cải tiến mới có thể báo `missing keys`; đây là bình thường vì layer mới học từ đầu.
- Quan hệ hình học giữa query có chi phí theo cặp query, nên nên để `BATCH_SIZE=1` trước.
- Chạy từng dataset nếu GPU time ít: `DATASETS="dior"`, `"dotav1_0"`, hoặc `"dotav1_5"`.
- Không visualize toàn bộ dataset. Script mặc định dùng `SAMPLE_IMAGES=10`.
- `/kaggle/working` sẽ mất khi session reset, nên cuối notebook có bước zip output.


## 1. Cấu hình Kaggle input và stage workflow

Bạn chỉ cần sửa các path `/kaggle/input/...` cho khớp tên Kaggle Dataset bạn attach. Các stage có thể bật/tắt bằng `RUN_*`.


In [ ]:
from pathlib import Path

# ===== Git source =====
# Kaggle sẽ clone source trực tiếp từ GitHub vào /kaggle/working/RQFormer.
REPO_GIT_URL = "https://github.com/hungnp-dev/RQFormer.git"
REPO_DIR = Path("/kaggle/working/RQFormer")

# ===== Dataset path =====
# Kaggle Dataset structure:
# /kaggle/input/datasets/jks010/rqformer-dataset/data/{DIOR, split_ss_dota, split_ss_dota1_5}
DATA_ROOT = Path("/kaggle/input/datasets/jks010/rqformer-dataset/data")
DATA_INPUTS = {
    "dior": DATA_ROOT / "DIOR",
    "dotav1_0": DATA_ROOT / "split_ss_dota",
    "dotav1_5": DATA_ROOT / "split_ss_dota1_5",
}

# ===== Optional checkpoint path =====
# Nếu bạn attach checkpoint riêng thì đặt folder ở đây. Nếu không có, workflow sẽ train từ đầu
# hoặc resume từ checkpoint trong /kaggle/working/work_dirs nếu đã có.
CKPT_INPUT = Path("/kaggle/input/rqformer-pth")
CKPT_DIR = Path("/kaggle/working/pth")

# ===== Working paths =====
WORK_ROOT = Path("/kaggle/working/work_dirs")

# ===== Dataset selection =====
# "all" hoặc danh sách cách nhau bởi dấu phẩy: "dior", "dotav1_0", "dotav1_5", "dior,dotav1_0"
DATASETS = "all"

# ===== Runtime controls =====
DEVICE = "0"
BATCH_SIZE = 1
NUM_WORKERS = 2
SAMPLE_IMAGES = 10
SCORE_THR = 0.3
FORCE = 0  # 1 = chạy lại stage đã done

# ===== Workflow stages =====
RUN_SETUP = 1
RUN_TRAIN = 1
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
RUN_SUMMARY = 1

print("REPO_GIT_URL =", REPO_GIT_URL)
print("DATA_ROOT =", DATA_ROOT)
print("DATASETS =", DATASETS)
print("WORK_ROOT =", WORK_ROOT)


## 2. Khai báo metadata 3 dataset/checkpoint/config

Phần này mirror đúng 3 job trong `tools/run_all_rqformer_visualize.sh`.


In [ ]:
JOBS = {
    "dior": {
        "title": "RQFormer | DIOR-R | R50 | 3x | Query 500 | t0.85",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth",
        "data_link": Path("data/DIOR"),
        "data_input": DATA_INPUTS["dior"],
    },
    "dotav1_0": {
        "title": "RQFormer | DOTA-v1.0 | R50 | 2x | Query 500 | t0.9",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.pth",
        "data_link": Path("data/split_ss_dota"),
        "data_input": DATA_INPUTS["dotav1_0"],
    },
    "dotav1_5": {
        "title": "RQFormer | DOTA-v1.5 | R50 | 2x | Query 500 | t0.9",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.pth",
        "data_link": Path("data/split_ss_dota1_5"),
        "data_input": DATA_INPUTS["dotav1_5"],
    },
}

def selected_names():
    if DATASETS == "all":
        return list(JOBS)
    names = [x.strip() for x in DATASETS.split(",") if x.strip()]
    unknown = [x for x in names if x not in JOBS]
    if unknown:
        raise ValueError(f"Unknown DATASETS: {unknown}. Valid: {list(JOBS)}")
    return names

print("Selected jobs:", selected_names())
for name in selected_names():
    print(name, "->", JOBS[name]["title"])


## 3. Copy source, link dataset, link checkpoint

Kaggle input là read-only. Cell này đưa source/checkpoint/data vào đúng cấu trúc mà script gốc đang cần:

- `REPO_DIR=/kaggle/working/RQFormer`
- `CKPT_DIR=/kaggle/working/pth`
- `data/DIOR`, `data/split_ss_dota`, `data/split_ss_dota1_5`


In [ ]:
## 3. Clone source, link dataset, link checkpoint

Kaggle input dataset là read-only, còn source code sẽ được clone từ GitHub:

```bash
git clone https://github.com/hungnp-dev/RQFormer.git /kaggle/working/RQFormer
```

Sau đó notebook tạo symlink dataset đúng cấu trúc mà config gốc cần:

- `/kaggle/working/RQFormer/data/DIOR -> /kaggle/input/datasets/jks010/rqformer-dataset/data/DIOR`
- `/kaggle/working/RQFormer/data/split_ss_dota -> /kaggle/input/datasets/jks010/rqformer-dataset/data/split_ss_dota`
- `/kaggle/working/RQFormer/data/split_ss_dota1_5 -> /kaggle/input/datasets/jks010/rqformer-dataset/data/split_ss_dota1_5`


import os
import shutil
import subprocess
from pathlib import Path


def clone_repo():
    if REPO_DIR.exists():
        print(f"[OK] Repo exists: {REPO_DIR}")
        return
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["git", "clone", REPO_GIT_URL, str(REPO_DIR)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)
    print(f"[OK] Cloned repo: {REPO_GIT_URL} -> {REPO_DIR}")


def link_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        print(f"[OK] Exists: {dst}")
        return
    if not src.exists():
        raise FileNotFoundError(f"Không thấy input: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
        print(f"[OK] Symlink: {dst} -> {src}")
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        print(f"[OK] Copied: {src} -> {dst}")


def prepare_data_and_ckpts():
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"Không thấy DATA_ROOT: {DATA_ROOT}")

    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    for name in selected_names():
        job = JOBS[name]
        link_or_copy(job["data_input"], REPO_DIR / job["data_link"])

        ckpt_src = CKPT_INPUT / job["ckpt"]
        ckpt_dst = CKPT_DIR / job["ckpt"]
        if ckpt_src.exists():
            link_or_copy(ckpt_src, ckpt_dst)
        else:
            print(f"[WARN] Không thấy checkpoint gốc: {ckpt_src}")
            print("       Nếu RUN_TEST=1 trước khi train xong thì stage test có thể skip do thiếu checkpoint.")

clone_repo()
os.chdir(REPO_DIR)
prepare_data_and_ckpts()
print("cwd =", Path.cwd())


In [ ]:
for name in selected_names():
    job = JOBS[name]
    cfg = REPO_DIR / job["config"]
    data = REPO_DIR / job["data_link"]
    print("\n[CHECK]", name)
    print("config:", cfg, cfg.exists())
    print("data:", data, data.exists())
    if not cfg.exists():
        raise FileNotFoundError(cfg)
    if not data.exists():
        raise FileNotFoundError(data)

script = REPO_DIR / "tools/run_all_rqformer_visualize.sh"
print("\nworkflow script:", script, script.exists())
if not script.exists():
    raise FileNotFoundError(script)


## 5. Chạy toàn bộ workflow bằng script gốc

Đây là cell chính. Nó gọi `bash tools/run_all_rqformer_visualize.sh` với biến môi trường Kaggle. Script sẽ tự skip stage đã done bằng các file `.done`, trừ khi `FORCE=1`.


In [ ]:
import subprocess
import sys


def run_workflow():
    env = os.environ.copy()
    env.update({
        "REPO_DIR": str(REPO_DIR),
        "CKPT_DIR": str(CKPT_DIR),
        "WORK_ROOT": str(WORK_ROOT),
        "CUDA_VISIBLE_DEVICES": str(DEVICE),
        "BATCH_SIZE": str(BATCH_SIZE),
        "NUM_WORKERS": str(NUM_WORKERS),
        "SAMPLE_IMAGES": str(SAMPLE_IMAGES),
        "SCORE_THR": str(SCORE_THR),
        "DATASETS": DATASETS,
        "FORCE": str(FORCE),
        "RUN_SETUP": str(RUN_SETUP),
        "RUN_TRAIN": str(RUN_TRAIN),
        "RUN_TEST": str(RUN_TEST),
        "RUN_CHARTS": str(RUN_CHARTS),
        "RUN_CONFUSION": str(RUN_CONFUSION),
        "RUN_BBOX": str(RUN_BBOX),
        "RUN_HEATMAP": str(RUN_HEATMAP),
        "RUN_SUMMARY": str(RUN_SUMMARY),
    })
    cmd = ["bash", "tools/run_all_rqformer_visualize.sh"]
    print("$", " ".join(cmd))
    print("DATASETS=", DATASETS)
    subprocess.check_call(cmd, cwd=REPO_DIR, env=env)

run_workflow()


## 6. Xem cấu trúc output sau khi chạy

Kết quả nằm dưới `work_dirs/rroiformer/<dataset>/` gồm train/test/charts/images/logs và `summary_results.md` tổng hợp.


In [ ]:
def tree(path: Path, max_depth=3):
    path = Path(path)
    if not path.exists():
        print("Not found:", path)
        return
    base_depth = len(path.parts)
    for p in sorted(path.rglob("*")):
        depth = len(p.parts) - base_depth
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        suffix = "/" if p.is_dir() else ""
        print(f"{indent}{p.name}{suffix}")

print("Output root:", WORK_ROOT)
tree(WORK_ROOT / "rroiformer", max_depth=4)


## 7. Đọc summary kết quả

Cột nào evaluator/log không có thì script summary sẽ để `-`, không tự thêm metric ngoài code/log.


In [ ]:
summary = WORK_ROOT / "rroiformer" / "summary_results.md"
if summary.exists():
    print(summary.read_text(encoding="utf-8", errors="replace"))
else:
    print("Chưa có summary:", summary)


## 8. Đóng gói output để tải về hoặc resume lần sau

Zip này chứa checkpoint, log, predictions, chart, bbox/heatmap mẫu và summary.


In [ ]:
archive_base = Path("/kaggle/working/rqformer2_full_workflow_outputs")
zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(WORK_ROOT))
print("Saved zip:", zip_path)


## 9. Các cách chạy nhanh

Mặc định chỉ cần chỉnh cell cấu hình đầu tiên rồi bấm **Run All**.

Chỉ setup + train DOTA-v1.0:

```python
DATASETS = "dotav1_0"
RUN_SETUP = 1
RUN_TRAIN = 1
RUN_TEST = 0
RUN_CHARTS = 0
RUN_CONFUSION = 0
RUN_BBOX = 0
RUN_HEATMAP = 0
RUN_SUMMARY = 0
```

Chỉ test/visualize từ checkpoint đã có:

```python
DATASETS = "dotav1_0"
RUN_SETUP = 0
RUN_TRAIN = 0
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
RUN_SUMMARY = 1
```

Chạy lại toàn bộ một dataset:

```python
DATASETS = "dior"
FORCE = 1
```
